# Create ESS instrument sources

In [ ]:
import mcstasscript as ms
import scipp as sc
from scipy.ndimage import gaussian_filter

In [ ]:
beamports = {
    "nmx": "W1",
    "beer": "W2",
    "cspec": "W3",
    "bifrost": "W4",
    "miracles": "W5",
    "magic": "W6",
    "trex": "W7",
    "heimdal": "W8",
    "odin": "S2",
    "dream": "S3",
    "loki": "N7",
    "freia": "N5",
    "estia": "E2",
    "skadi": "E3",
    "vespa": "E7",
    "tbl": "W11",
}

In [ ]:
for instr, port in beamports.items():

    print("Making source for", instr, port)

    instrument = ms.McStas_instr("source_check")

    source = instrument.add_component("Source", "ESS_butterfly")
    # different spectra for each sector/beamport
    source.set_parameters(sector=f'"{port[0]}"', beamline=int(port[1]), n_pulses=1)
    source.tmax_multiplier = 3.0 # default 3, samples the tail out to 3 x 2.86 ms
    source.acc_power = 2 # accelerator power in MW
    source.set_parameters(Lmin=0.1, Lmax=20) # wavelength range
    # investigate rays that go to this place, 2 m in front, 10x10 cm area (distribution differs depending on direction)
    source.set_parameters(focus_xw=0.1, focus_yh=0.1, dist=2)

    # monitor = instrument.add_component("surface_monitor", "Monitor_nD")
    # monitor.set_parameters(options='"previous, x y z vx vy vz l t list all neutrons"', restore_neutron=1)

    monitor = instrument.add_component("flat_monitor", "Monitor_nD")
    monitor.set_parameters(xwidth=1.0, yheight=1.0, restore_neutron=1) # large flat monitor
    monitor.options=f'"square, l bins=1024 limits=[{source.Lmin} {source.Lmax}] t bins=1024 limits=[0 {source.tmax_multiplier*2E-3}]"'
    monitor.set_AT(0.05, RELATIVE=source)

    instrument.settings(ncount=1.0e8, mpi=8)

    data = instrument.backengine()

    smooth = gaussian_filter(data[0].Intensity.T, sigma=2)

    lims = data[0].metadata.limits

    da = sc.DataArray(
        data = sc.array(dims=['wavelength', 'birth_time'], values=smooth/smooth.sum()),
        coords={"birth_time": sc.midpoints(sc.linspace('birth_time', lims[2], lims[3], smooth.shape[1] + 1, unit='s').to(unit='us')),
                "wavelength": sc.midpoints(sc.linspace('wavelength', lims[0], lims[1], smooth.shape[0] + 1, unit='angstrom')),
                "frequency": sc.scalar(14.0, unit="Hz"),
                "distance": sc.scalar(0.05, unit="m")}
    )

    da.save_hdf5(f'ess-{instr}.h5')

## Default ESS source

To create the default ESS source, we simply take the mean profile across all instruments.

In [ ]:
all_sources = sc.concat([sc.io.load_hdf5(f'ess-{instr}.h5') for instr in beamports], dim='instrument')
mean_source = all_sources.mean('instrument')
mean_source = mean_source / mean_source.sum()
mean_source.save_hdf5('ess.h5')

In [ ]:
mean_source.plot()